# Lab Task – Simple RAG System for Tips Hindawi University

This notebook follows the same overall pipeline used in `Lecture_3_RAG.ipynb`:

1. Install/import libraries.
2. Load an instruction-tuned causal language model.
3. Define a text-generation function.
4. Extract text from the PDF.
5. Split the PDF text into chunks.
6. Convert chunks to embeddings.
7. Build a FAISS vector index.
8. Retrieve the most relevant chunks for a question.
9. Put the retrieved context into a prompt.
10. Generate the final answer.

**Bilingual design:** Qwen2.5-3B-Instruct is used as the generator and `intfloat/multilingual-e5-base` as the embedding model so the same RAG pipeline can accept English and Arabic questions. The PDF used below is the English `Tips Hindawi University Info.pdf`. An Arabic PDF can be substituted later without changing the RAG structure.

In [1]:
# Install the same main libraries used by the lecture notebook.
# Qwen2.5-3B-Instruct is used instead of Mistral-Nemo because we want
# one generator that can work well with both English and Arabic.

!pip install -q transformers==4.52.4 sentence-transformers PyPDF2 faiss-cpu accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 75.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 80.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 71.7 MB/s eta 0:00:00:00:01


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader
import torch
import faiss
import os

In [3]:
# Bilingual instruction-following generator.
# Qwen2.5-3B-Instruct

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [4]:
def generate_text(prompt, max_new_tokens=250):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a retrieval-augmented question answering assistant. "
                "Answer ONLY from the supplied context. "
                "If the context does not contain the answer, say that the information "
                "is not specified in the document. Never invent facts."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id
    )

    # Return only newly generated tokens, not the original prompt.
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [6]:
# PDF text extraction – same idea as the lecture notebook.

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""

    for page in reader.pages:
        page_text = page.extract_text() or ""
        full_text += page_text + "\n"

    return full_text

In [7]:
# Text chunking – same word-based approach as the lecture notebook.
# The smaller chunk size is useful for this relatively short university PDF.

def chunk_text(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])

        if chunk.strip():
            chunks.append(chunk)

    return chunks

In [8]:
# Multilingual embeddings.
# multilingual-e5-base supports many languages, including Arabic and English.

def embed_chunks(chunks, model_name="intfloat/multilingual-e5-base"):
    embedding_model = SentenceTransformer(model_name)

    # E5 recommends "passage:" for stored document chunks.
    passage_texts = [f"passage: {chunk}" for chunk in chunks]

    embeddings = embedding_model.encode(
        passage_texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return embedding_model, embeddings

In [9]:
def create_faiss_index(embeddings):
    # Because embeddings are normalized, inner product is equivalent
    # to cosine similarity.

    dim = embeddings.shape[1]

    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    return index

In [10]:
def search_index(query, model, index, chunks, k=3):
    # E5 recommends "query:" for user questions.
    query_embedding = model.encode(
        [f"query: {query}"],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            results.append({
                "chunk": chunks[idx],
                "score": float(score)
            })

    return results

In [11]:
# Path to the uploaded PDF.

pdf_path = "/kaggle/input/datasets/andrewsameh20/tips-hindawi-university-information/Tips Hindawi University Info.pdf"

text = extract_text_from_pdf(pdf_path)

chunks = chunk_text(
    text,
    chunk_size=100,
    overlap=20
)

model_embeddings, embeddings = embed_chunks(chunks)

index = create_faiss_index(embeddings)

print("Number of characters:", len(text))
print("Number of chunks:", len(chunks))
print("Embedding dimension:", embeddings.shape[1])

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Number of characters: 4112
Number of chunks: 8
Embedding dimension: 768


In [12]:
# Optional inspection: print the first few chunks.

for i, chunk in enumerate(chunks[:5], 1):
    print(f"\n--- Chunk {i} ---\n{chunk}")


--- Chunk 1 ---
1. General Overview Tips Hindawi University (THU) is a premier institution of higher education located in the heart of the Middle East. Founded in 1963, the university has grown into a globally recognized center for academic excellence and innovation. With over six decades of educational leadership, THU has produced more than 150,000 graduates who serve in diverse industries and academic circles worldwide. The university is accredited by the International Commission for Academic Standards and the Ministry of Higher Education. It operates under the guiding motto: "Knowledge, Integrity, Progress" , and fosters an environment of research, creativity, and public service. 2.

--- Chunk 2 ---
operates under the guiding motto: "Knowledge, Integrity, Progress" , and fosters an environment of research, creativity, and public service. 2. Campus and Facilities 2.1 Main Campus The main campus is located in the capital city and spans over 300 acres. It houses: - 12 academic buildin

In [13]:
def answer_question(question, k=3, max_new_tokens=250):
    # Retrieve the most relevant chunks.
    top_chunks = search_index(
        question,
        model_embeddings,
        index,
        chunks,
        k=k
    )

    # Combine retrieved chunks into the RAG context.
    context = "\n\n".join(
        [
            f"[Retrieved Chunk {i+1}]\n{item['chunk']}"
            for i, item in enumerate(top_chunks)
        ]
    )

    # question + retrieved text -> prompt -> generator.
    prompt = f"""Answer the following question using ONLY the retrieved context.

Question:
{question}

Retrieved context:
{context}

Rules:
- Use only the retrieved context.
- Do not add outside knowledge.
- If the answer is not explicitly supported by the context, say:
  "The information is not specified in the document."
- Answer in the same language as the question.
"""

    answer = generate_text(
        prompt,
        max_new_tokens=max_new_tokens
    )

    return answer, top_chunks, prompt

## Questions Required by the Lab

In [14]:
questions = ['Where is Tips Hindawi University located?', 'Does the university offer online programs?', 'Is there financial aid for international students?', 'What languages are used for instruction?']

for q in questions:
    print(f"\n{'='*80}\nQUESTION: {q}\n{'='*80}")
    answer, top_chunks, prompt = answer_question(q, k=3)
    print(answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



QUESTION: Where is Tips Hindawi University located?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The main campus of Tips Hindawi University is located in the capital city and spans over 300 acres.

QUESTION: Does the university offer online programs?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The information is not specified in the document.

QUESTION: Is there financial aid for international students?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The information is not specified in the document.

QUESTION: What languages are used for instruction?
The information is not specified in the document.


In [15]:
# Inspect retrieved evidence for each question.
# This is useful for demonstrating that the generator is grounded in the PDF.

for q in questions:
    answer, top_chunks, prompt = answer_question(q, k=3)

    print(f"\n\nQUESTION: {q}")
    print("ANSWER:", answer)

    for i, item in enumerate(top_chunks, 1):
        print(f"\n--- Retrieved Chunk {i} | Score: {item['score']:.4f} ---")
        print(item["chunk"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.




QUESTION: Where is Tips Hindawi University located?
ANSWER: The main campus of Tips Hindawi University is located in the capital city and spans over 300 acres.

--- Retrieved Chunk 1 | Score: 0.8425 ---
1. General Overview Tips Hindawi University (THU) is a premier institution of higher education located in the heart of the Middle East. Founded in 1963, the university has grown into a globally recognized center for academic excellence and innovation. With over six decades of educational leadership, THU has produced more than 150,000 graduates who serve in diverse industries and academic circles worldwide. The university is accredited by the International Commission for Academic Standards and the Ministry of Higher Education. It operates under the guiding motto: "Knowledge, Integrity, Progress" , and fosters an environment of research, creativity, and public service. 2.

--- Retrieved Chunk 2 | Score: 0.7517 ---
• • • • • • • • • • • • • • • 3 9. Alumni and Impact Lina Darwish : UN Yo

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.




QUESTION: Does the university offer online programs?
ANSWER: The information is not specified in the document.

--- Retrieved Chunk 1 | Score: 0.7742 ---
: Prof. Layla Mahmoud Dean of Students : Mr . Samer Hussein Registrar's Office : Handles academic records, course registration, transcripts Office of International Programs : Assists international students and partnerships 6. Research and Innovation Center for Renewable Energy Research Artificial Intelligence and Robotics Lab Public Policy and Development Institute Annual "Future Frontiers" Conference Research budget: $12 million annually Sample Project: - "Solar-Powered Desalination Systems for Arid Regions" led by Prof. Amina Al-Rawi 7. Student Life 7.1 Clubs and Organizations THU Debate Society Robotics Club Arabic Calligraphy Circle Environmental Awareness Network International Students Union 7.2 Events and Traditions Founders' Week

--- Retrieved Chunk 2 | Score: 0.7600 ---
operates under the guiding motto: "Knowledge, Integrit

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.




QUESTION: Is there financial aid for international students?
ANSWER: The information is not specified in the document.

--- Retrieved Chunk 1 | Score: 0.7965 ---
: Prof. Layla Mahmoud Dean of Students : Mr . Samer Hussein Registrar's Office : Handles academic records, course registration, transcripts Office of International Programs : Assists international students and partnerships 6. Research and Innovation Center for Renewable Energy Research Artificial Intelligence and Robotics Lab Public Policy and Development Institute Annual "Future Frontiers" Conference Research budget: $12 million annually Sample Project: - "Solar-Powered Desalination Systems for Arid Regions" led by Prof. Amina Al-Rawi 7. Student Life 7.1 Clubs and Organizations THU Debate Society Robotics Club Arabic Calligraphy Circle Environmental Awareness Network International Students Union 7.2 Events and Traditions Founders' Week

--- Retrieved Chunk 2 | Score: 0.7861 ---
• • • • • • 1 3.2 Sample Undergraduate Program